Урок в прозе: https://proproprogs.ru/python_oop/python-mnozhestvennoe-nasledovanie

Телеграм-канал: https://t.me/python_selfedu

Один дочерний класс может быть унаследован сразу от нескольких базовых классов.

In [1]:
class Goods:
    def __init__(self, name, weight, price):
        print("init RixinLog")
        self.name = name
        self.weight = weight
        self.price = price
    def print_info(self):
        print(f"{self.name}, {self.weight}, {self.price}")

class NoteBook(Goods):
    pass

n = NoteBook('acer', 1.5, 33333)
n.print_info()

init RixinLog
acer, 1.5, 33333


Вдруг нам понадобилась возможность логирования товаров. Это можно реализовать через текущий класс или через базовый класс. А можно реализовать через mixins (это такой паттерн) (примесь)

In [3]:
class Goods:
    def __init__(self, name, weight, price):
        print("init Goods")
        self.name = name
        self.weight = weight
        self.price = price
    def print_info(self):
        print(f"{self.name}, {self.weight}, {self.price}")


class MixinLog:
    ID = 0
    def __init__(self):
        print('init MixinLog')
        self.ID += 1
        self.id = self.ID

    def save_sell_log(self):
        print(f'{self.id: товар был продан}')


class NoteBook(Goods, MixinLog):
    pass

n = NoteBook('acer', 1.5, 33333)
n.print_info()
n.save_sell_log()

init Goods
acer, 1.5, 33333


AttributeError: 'NoteBook' object has no attribute 'id'

__init__ класса MixinLog не отработал. Как мы знаем, __init__ сначала ищется в текущем классе, потом в первом базовом из указанных.

In [6]:
class Goods:
    def __init__(self, name, weight, price):
        super().__init__() # каким-то образом здесь происходит обращение к классу MixinLog
        print("init Goods")
        self.name = name
        self.weight = weight
        self.price = price
    def print_info(self):
        print(f"{self.name}, {self.weight}, {self.price}")


class MixinLog:
    ID = 0
    def __init__(self):
        print('init MixinLog')
        self.ID += 1
        self.id = self.ID

    def save_sell_log(self):
        print(f'{self.id}: товар был продан')


class NoteBook(Goods, MixinLog):
    pass

n = NoteBook('acer', 1.5, 33333)
n.print_info()
n.save_sell_log()

init MixinLog
init Goods
acer, 1.5, 33333
1: товар был продан


Но каким образом super().__init__() внутри базового класса Goods обращается к MixinLog, а не к object?? Дело в том, что есть алгоритм обхода базовых классов при множественном наследовании MRO (method resolution order): сначала метод () ищется в текущем классе, потом его базовых классах по порядку их перечисления в объевлении текущего, и потом в object. Но для этого в первом базовом классе должна быть ссылка через super()

In [7]:
NoteBook.__mro__

(__main__.NoteBook, __main__.Goods, __main__.MixinLog, object)

Дополнительные классы в mixins не должны в ините иметь никаких аргументов, кроме self, иначе будут сложности с использованием

In [17]:
class Goods:
    def __init__(self, name, weight, price):
        super().__init__(1) # согласно MRO здесь происходит обращение к классу MixinLog
        print("init Goods")
        self.name = name
        self.weight = weight
        self.price = price
    def print_info(self):
        print(f"{self.name}, {self.weight}, {self.price}")


class MixinLog:
    ID = 0
    def __init__(self, p1): # ввели новый аргумент p1
        super().__init__(1, 2) # p1 и p2
        print('init MixinLog')
        self.ID += 1
        self.id = self.ID

    def save_sell_log(self):
        print(f'{self.id}: товар был продан')

class MixinLog2:
    def __init__(self, p1, p2): # ввели новые аргументы p1 и p2
        # super().__init__()
        print('init MixinLog2')



class NoteBook(Goods, MixinLog, MixinLog2):
    pass

n = NoteBook('acer', 1.5, 3333)
n.print_info()
n.save_sell_log()

init MixinLog2
init MixinLog
init Goods
acer, 1.5, 3333
1: товар был продан


Чтобы это работало, миксины должны быть перечислены в строгом порядке, а этот порядок учтен в обращениях super().__init__() в инициализаторе каждого миксина. Чтобы за этим не требовалось следить, принято инициализаторы миксинов использовать без параметров

In [23]:
class Goods:
    def __init__(self, name, weight, price):
        super().__init__() # согласно MRO здесь происходит обращение к классу MixinLog
        print("init Goods")
        self.name = name
        self.weight = weight
        self.price = price
    def print_info(self):
        print(f"{self.name}, {self.weight}, {self.price}")


class MixinLog:
    ID = 0
    def __init__(self):
        super().__init__()
        print('init MixinLog')
        self.ID += 1
        self.id = self.ID

    def save_sell_log(self):
        print(f'{self.id}: товар был продан')

class MixinLog2:
    def __init__(self):
        super().__init__() # здесь нужно оставить ссылку, на случай если она должна вести на следующий по списку базовый класс
        print('init MixinLog2')



class NoteBook(Goods, MixinLog2, MixinLog ):
    pass

n = NoteBook('acer', 1.5, 3333)
NoteBook.__mro__

init MixinLog
init MixinLog2
init Goods


(__main__.NoteBook,
 __main__.Goods,
 __main__.MixinLog2,
 __main__.MixinLog,
 object)

Теперь об использовании методов с одинаковыми именами

In [26]:
class Goods:
    def __init__(self, name, weight, price):
        super().__init__() # согласно MRO здесь происходит обращение к классу MixinLog
        # print("init Goods")
        self.name = name
        self.weight = weight
        self.price = price
    def print_info(self):
        print(f"вызван print_info из Goods")


class MixinLog:
    ID = 0
    def __init__(self):
        super().__init__()
        # print('init MixinLog')
        self.ID += 1
        self.id = self.ID

    def print_info(self):
        print(f"вызван print_info из MixinLog")

class NoteBook(Goods, MixinLog):
    pass

n = NoteBook('acer', 1.5, 3333)
n.print_info()

вызван print_info из Goods


Вызвать одноименный метод из MixinLog можно напрямую

In [27]:
MixinLog.print_info(n)

вызван print_info из MixinLog


Но если нам конкретно нужно, чтобы метод вызывался из второго базового класса, это можно переопределить в дочернем классе

In [28]:
class Goods:
    def __init__(self, name, weight, price):
        super().__init__() # согласно MRO здесь происходит обращение к классу MixinLog
        self.name = name
        self.weight = weight
        self.price = price
    def print_info(self):
        print(f"вызван print_info из Goods")


class MixinLog:
    ID = 0
    def __init__(self):
        super().__init__()
        self.ID += 1
        self.id = self.ID

    def print_info(self):
        print(f"вызван print_info из MixinLog")

class NoteBook(Goods, MixinLog):
    def print_info(self):
        MixinLog.print_info(self)

n = NoteBook('acer', 1.5, 3333)
n.print_info()

вызван print_info из MixinLog
